# Curation case study
## Increase quality score, step by step

This is the pre-publication workflow chemcheck is built for: **audit → fix → re-audit**, where every fix is a response to a specific finding. We reuse the dirty dataset from notebook 01 and curate it with plain RDKit + pandas:

| step | finding it answers |
|---|---|
| 1. Drop unparseable rows | `invalid_smiles`, `valence_error` |
| 2. Standardize (desalt + tautomers) | `salt_duplicates`, `tautomer_duplicates` |
| 3. Aggregate replicates | `duplicated_measurements`, `conflicting_measurements` |
| 4. Collapse stereoisomers | `stereochemical_collisions`, `identity_leakage` |
| 5. Regroup the alkyl series | `cross_split_near_duplicates`, `analog_leakage` |

In [1]:
import pandas as pd

rows = [
    ("d01", "CCO",            1.2, "train"), ("d02", "CCO",            1.2, "train"),
    ("d03", "c1ccccc1",       2.0, "train"), ("d04", "C[C@H](Cl)Br",   3.0, "train"),
    ("d05", "C[C@@H](Cl)Br",  3.5, "test"),  ("d06", "CC(Cl)Br",       3.1, "test"),
    ("d07", "CCO.[Na+]",       1.2, "test"),  ("d08", "not_a_smiles!!", 0.0, "train"),
    ("d09", "C(C)(C)(C)(C)C", 0.5, "train"), ("d10", "[C-2]",           0.1, "train"),
    ("d11", "CC(=O)C",        4.0, "train"), ("d12", "CC(O)=C",        5.5, "train"),
    ("d13", "[13C]C",         1.0, "train"), ("d14", "c1ccccc1O",      2.5, "test"),
    ("d15", "c1ccccc1O",      7.9, "test"),  ("d16", "CCCCCCCCO",      1.1, "train"),
    ("d17", "CCCCCCCCCO",     1.3, "test"),  ("d18", "CC(=O)O",        9.9, "train"),
    ("d19", "[U]C",           0.2, "train"), ("d20", "C1OC1CCl",       1.4, "test"),
]
df = pd.DataFrame(rows, columns=["id", "smiles", "activity", "split"])
df.to_csv("dirty.csv", index=False)

from chemcheck.report import audit
rep0 = audit(["dirty.csv"], label_col="activity", split_col="split")
err0 = sorted({f.check_id for f in rep0.findings if f.severity.value == "error"})
print(f"baseline: score {rep0.score}/100, n={rep0.n_total}")
print("errors:", err0)

baseline: score 39/100, n=20
errors: ['conflicting_measurements', 'cross_split_near_duplicates', 'identity_leakage', 'invalid_smiles', 'valence_error']


## Step 1: drop what cannot be parsed

`invalid_smiles` and `valence_error` rows are not data: they are parse failures that silently shrink (and bias) your training set. A row RDKit cannot sanitize with default settings goes:

In [2]:
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog("rdApp.*")

parseable = df["smiles"].apply(lambda s: Chem.MolFromSmiles(str(s)) is not None)
print("dropped:", df.loc[~parseable, ["id", "smiles"]].to_dict("records"))
work = df[parseable].copy().reset_index(drop=True)
print(f"{len(df)} → {len(work)} rows")

dropped: [{'id': 'd08', 'smiles': 'not_a_smiles!!'}, {'id': 'd09', 'smiles': 'C(C)(C)(C)(C)C'}]
20 → 18 rows


## Step 2: standardize. Desalt, then canonicalize tautomers

One function, applied to every row. Note the defensive pattern: **standardizers are not total functions**. Watch what RDKit's default salt stripper does to neutral acetic acid (`CC(=O)O`): carboxylic acids match its salt list, so `StripMol` returns an *empty* molecule. If we didn't check each stage's output, we'd delete good data. Always validate transformation outputs (and the re-audit at the end would catch it anyway):

In [3]:
from rdkit.Chem import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

_rem = SaltRemover.SaltRemover()
_te = rdMolStandardize.TautomerEnumerator()

# the quirk, isolated:
q = Chem.MolToSmiles(_rem.StripMol(Chem.MolFromSmiles("CC(=O)O")), isomericSmiles=True)
print(f"desalted acetic acid: {q!r}  <- the stripper ate a neutral acid! fall back to input")

def standardize(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    step = Chem.MolToSmiles(mol, isomericSmiles=True)
    try:
        s = Chem.MolToSmiles(_rem.StripMol(mol), isomericSmiles=True)
        if s:
            step = s                                     # d07: CCO.[Na+] -> CCO
    except Exception:
        pass
    try:
        s = Chem.MolToSmiles(_te.Canonicalize(Chem.MolFromSmiles(step)),
                             isomericSmiles=True)
        if s:
            step = s                                     # d11/d12: keto + enol -> one form
    except Exception:
        pass
    return step or None

work["std"] = work["smiles"].apply(standardize)
assert work["std"].notna().all(), "standardizer produced gaps — investigate before continuing"
print(work.loc[work["smiles"] != work["std"], ["id", "smiles", "std"]].to_string(index=False))

desalted acetic acid: ''  <- the stripper ate a neutral acid! fall back to input
 id    smiles       std
d07 CCO.[Na+]       CCO
d11   CC(=O)C   CC(C)=O
d12   CC(O)=C   CC(C)=O
d13    [13C]C    C[13C]
d14 c1ccccc1O Oc1ccccc1
d15 c1ccccc1O Oc1ccccc1
d19      [U]C  [CH3][U]
d20  C1OC1CCl  ClCC1CO1


## Step 3: aggregate replicates and conflicts

After standardization, `CCO` appears three times (d01, d02, desalted d07) and phenol twice with *contradictory* labels (2.5 vs 7.9). A model cannot learn contradictory labels. Average replicates into one row per structure (here a plain mean; in production, weight by assay quality):

In [4]:
work["connectivity"] = work["std"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s), isomericSmiles=False))

agg = (work.groupby("std")
             .agg(id=("id", "first"),
                  activity=("activity", "mean"),      # replicates averaged, conflicts surfaced
                  split=("split", "first"),
                  connectivity=("connectivity", "first"))
             .reset_index())
print(f"{len(work)} → {len(agg)} unique structures")
print("phenol conflict resolved to mean:",
      round(float(agg.loc[agg["std"] == "Oc1ccccc1", "activity"].iloc[0]), 2))

18 → 14 unique structures
phenol conflict resolved to mean: 5.2


## Step 4: collapse stereoisomers

`C[C@H](Cl)Br` / `C[C@@H](Cl)Br` / `CC(Cl)Br` share one connectivity but span the train/test split. This is stereo-agnostic identity leakage. Unless your endpoint is stereo-dependent, keep one row per connectivity (here: the first, i.e. the train copy):

In [5]:
ded = agg.drop_duplicates("connectivity", keep="first").copy().reset_index(drop=True)
print(f"{len(agg)} → {len(ded)} rows (stereo trio collapsed to one)")

14 → 12 rows (stereo trio collapsed to one)


## Step 5: regroup the chemical series

Octanol (train) and nonanol (test) are Morgan-identical (Tc = 1.0): the same alkyl series on both sides of the split. Move nonanol to train so the series lives in one split:

In [6]:
ded.loc[ded["std"] == "CCCCCCCCCO", "split"] = "train"
ded.to_csv("curated.csv", index=False)
print(ded[["id", "std", "activity", "split"]].to_string(index=False))

 id        std  activity split
d18    CC(=O)O      9.90 train
d11    CC(C)=O      4.75 train
d06   CC(Cl)Br      3.10  test
d17 CCCCCCCCCO      1.30 train
d16  CCCCCCCCO      1.10 train
d01        CCO      1.20 train
d13     C[13C]      1.00 train
d20   ClCC1CO1      1.40  test
d14  Oc1ccccc1      5.20  test
d10      [C-2]      0.10 train
d19   [CH3][U]      0.20 train
d03   c1ccccc1      2.00 train


## Re-audit: did it work?

In [7]:
rep1 = audit(["curated.csv"], smiles_col="std", id_col="id",
             label_col="activity", split_col="split")
err1 = sorted({f.check_id for f in rep1.findings if f.severity.value == "error"})
warn1 = sorted({f.check_id for f in rep1.findings if f.severity.value == "warning"})
print(f"score: {rep0.score} → {rep1.score}   (n={rep0.n_total} → {rep1.n_total})")
print("errors left:", err1 or "none 🎉")
print("warnings left:", warn1)

score: 39 → 80   (n=20 → 12)
errors left: none 🎉
warnings left: ['impossible_charge', 'near_duplicates', 'radical_flag', 'scaffold_overlap', 'unspecified_stereo']


The remaining warnings are **documented judgment calls**, not oversights — this is what you put in the paper's data section:

- `impossible_charge` / `radical_flag` — `[C-2]` and `[U]C` are exotic but intentional here; flag the subset for readers.
- `unspecified_stereo` — the epoxide's center is genuinely unknown; evaluate stereo-agnostically.
- `near_duplicates` / `scaffold_overlap` — the alkyl series now lives in train (fine), and benzene is shared across splits (report the regime, cf. notebook 02).

## Export the report as a supplementary artifact

In [8]:
from chemcheck.report import render_html

with open("curated_report.html", "w") as fh:
    fh.write(render_html(rep1))
print("wrote curated_report.html — attach it to the submission as the dataset QC record")

# the notebook's tests: curation must remove all errors and substantially raise the score
assert not err1, err1
assert {"invalid_smiles", "conflicting_measurements", "identity_leakage",
        "cross_split_near_duplicates"} <= (set(err0) - set(err1))
assert rep1.score >= rep0.score + 40, (rep0.score, rep1.score)
print(f"curation checks passed ({rep0.score} → {rep1.score}, zero errors)")

wrote curated_report.html — attach it to the submission as the dataset QC record
curation checks passed (39 → 80, zero errors)
